In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import rc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
import imageio.v2 as imageio

# 设置输出目录
output_parent = "/home/weiji/restart_exam/20250221/polar_vortex_analysis_G/wind_Z3"
os.makedirs(output_parent, exist_ok=True)

# 定义要分析的年份和月份
years = ['0008', '0013', '0014', '0019']
months = ['Feb', 'Mar']

# 定义压力层（单位 hPa）
pressure_levels = [10, 50]

In [ ]:
#########################################
# 函数: load_experiment_field_for_var
#########################################
def load_experiment_field_for_var(file_pattern, var_name, lat_min=60, lat_max=90):
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    data = ds[var_name]
    data = data.sel(lat=slice(lat_min, lat_max))
    if 'member' in data.dims:
        data = data.mean(dim='member')
    return data

#########################################
# 加载 'nocoupl' 数据
#########################################
def load_year_nocouple_field_var(year, month, var_name, lat_min=60, lat_max=90):
    if month == 'Feb':
        file_pattern = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    elif month == 'Mar':
        file_pattern = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    else:
        raise ValueError("Invalid month")
    return load_experiment_field_for_var(file_pattern, var_name, lat_min, lat_max)

#########################################
# 加载 'small' 数据
#########################################
def load_year_small_pert_field_var(year, month, var_name, lat_min=60, lat_max=90):
    if month == 'Feb':
        file_pattern = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}/Feb/BWCN.e122.f19_g16.002.{year}-02.*.nc'
    elif month == 'Mar':
        file_pattern = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}/Mar/BWCN.e122.f19_g16.002.{year}-03.*.nc'
    else:
        raise ValueError("Invalid month")
    return load_experiment_field_for_var(file_pattern, var_name, lat_min, lat_max)

In [ ]:
#########################################
# 函数: plot_composite_daily_maps_wind_Z3
#########################################
def plot_composite_daily_maps_wind_Z3(exp1_name, Z3_data1, U_data1, V_data1,
                                      exp2_name, Z3_data2, U_data2, V_data2,
                                      output_dir, pressure_level, frame_duration=500):
    """
    对每个时间步，创建一个北极立体投影图：
      - 计算风速（sqrt(U^2+V^2)），并以填色图方式绘制
      - 叠加 geopotential height (Z3, 单位 km) 的等值线
    """
    os.makedirs(output_dir, exist_ok=True)
    rc('font', **{'family': 'sans-serif', 'sans-serif': ['Helvetica']})
    rc('text', usetex=False)

    target_pressure = pressure_level * 100

    Z3_data1_level = Z3_data1.sel(plev=target_pressure, method='nearest')
    U_data1_level = U_data1.sel(plev=target_pressure, method='nearest')
    V_data1_level = V_data1.sel(plev=target_pressure, method='nearest')
    
    Z3_data2_level = Z3_data2.sel(plev=target_pressure, method='nearest')
    U_data2_level = U_data2.sel(plev=target_pressure, method='nearest')
    V_data2_level = V_data2.sel(plev=target_pressure, method='nearest')

    n_frames = min(Z3_data1_level.time.size, Z3_data2_level.time.size)

    for i in range(n_frames):
        # 提取当前时刻数据
        Z3_1 = Z3_data1_level.isel(time=i)
        U_1 = U_data1_level.isel(time=i)
        V_1 = V_data1_level.isel(time=i)
        
        Z3_2 = Z3_data2_level.isel(time=i)
        U_2 = U_data2_level.isel(time=i)
        V_2 = V_data2_level.isel(time=i)

        # 计算风速，并为风速和 Z3 添加循环点（Z3 转换为 km）
        wind_speed_1 = np.sqrt(U_1.values**2 + V_1.values**2)
        wind_speed_2 = np.sqrt(U_2.values**2 + V_2.values**2)
        wind_speed_1_cyclic, lons = add_cyclic_point(wind_speed_1, coord=Z3_1.lon.values)
        wind_speed_2_cyclic, _ = add_cyclic_point(wind_speed_2, coord=Z3_2.lon.values)
        Z3_1_km_cyclic, _ = add_cyclic_point(Z3_1.values/1000., coord=Z3_1.lon.values)
        Z3_2_km_cyclic, _ = add_cyclic_point(Z3_2.values/1000., coord=Z3_2.lon.values)

        # 处理时间标签
        time_val = Z3_1.time.values
        if isinstance(time_val, np.ndarray) and time_val.size == 1:
            time_val = time_val.item()
        time_val = np.datetime64(time_val)
        time_label = np.datetime_as_string(time_val, unit='D')

        fig, axes = plt.subplots(1, 2, subplot_kw={'projection': ccrs.NorthPolarStereo()}, figsize=(16,8))
        for ax in axes:
            ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
            ax.coastlines()
            ax.add_feature(cfeature.LAND, facecolor='lightgray')
            gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray')
            gl.xlabels_top = False
            gl.ylabels_right = False

        # 根据压力层设置风速色标范围
        if pressure_level == 50:
            wind_levels = np.linspace(0, 40, 41)
            wind_vmin, wind_vmax = 0, 40
        else:
            wind_levels = np.linspace(0, 150, 31)
            wind_vmin, wind_vmax = 0, 150

        # 左侧面板：实验 1
        cs1 = axes[0].contourf(lons, Z3_1.lat, wind_speed_1_cyclic, levels=wind_levels,
                               cmap='viridis', vmin=wind_vmin, vmax=wind_vmax, transform=ccrs.PlateCarree())
        cs1_lines = axes[0].contour(lons, Z3_1.lat, Z3_1_km_cyclic, levels=10, colors='black',
                                    linewidths=1.0, transform=ccrs.PlateCarree())
        axes[0].clabel(cs1_lines, fmt='%.0f km', inline=True, fontsize=10)
        axes[0].set_title(f"{exp1_name} (Wind Speed & Z3) at {pressure_level} hPa\nDate: {time_label}", fontsize=14)

        # 右侧面板：实验 2
        cs2 = axes[1].contourf(lons, Z3_2.lat, wind_speed_2_cyclic, levels=wind_levels,
                               cmap='viridis', vmin=wind_vmin, vmax=wind_vmax, transform=ccrs.PlateCarree())
        cs2_lines = axes[1].contour(lons, Z3_2.lat, Z3_2_km_cyclic, levels=10, colors='black',
                                    linewidths=1.0, transform=ccrs.PlateCarree())
        axes[1].clabel(cs2_lines, fmt='%.0f km', inline=True, fontsize=10)
        axes[1].set_title(f"{exp2_name} (Wind Speed & Z3) at {pressure_level} hPa\nDate: {time_label}", fontsize=14)

        cbar = fig.colorbar(cs2, ax=axes, orientation='horizontal', fraction=0.05, pad=0.08)
        cbar.set_label(f"Wind Speed (m/s)", fontsize=12)

        out_fname = os.path.join(output_dir, f"frame_{i+1:03d}.png")
        plt.savefig(out_fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved frame {i+1} to {out_fname}")

    files = sorted([f for f in os.listdir(output_dir) if f.endswith('.png')])
    if files:
        images = []
        for fname in files:
            fpath = os.path.join(output_dir, fname)
            try:
                images.append(imageio.imread(fpath))
            except Exception as e:
                print(f"Error reading {fpath}: {e}")
        gif_fname = os.path.join(output_dir, f"composite_wind_Z3_{pressure_level}hPa.gif")
        imageio.mimsave(gif_fname, images, duration=frame_duration/1000)
        print(f"Saved GIF: {gif_fname}")
    else:
        print(f"No PNG files found in {output_dir}!")

In [ ]:
#########################################
# 主处理循环
#########################################
for year in years:
    for month in months:
        print(f"Processing Wind & Z3 for Year: {year}, Month: {month}")
        Z3_nocoup = load_year_nocouple_field_var(year, month, 'Z3')
        U_nocoup = load_year_nocouple_field_var(year, month, 'U')
        V_nocoup = load_year_nocouple_field_var(year, month, 'V')
        
        Z3_small = load_year_small_pert_field_var(year, month, 'Z3')
        U_small = load_year_small_pert_field_var(year, month, 'U')
        V_small = load_year_small_pert_field_var(year, month, 'V')
        
        for p in pressure_levels:
            print(f"Processing pressure level: {p} hPa")
            out_dir = os.path.join(output_parent, f"{year}_{month}_{p}hPa")
            os.makedirs(out_dir, exist_ok=True)
            plot_composite_daily_maps_wind_Z3("nocoupl", Z3_nocoup, U_nocoup, V_nocoup,
                                            "small", Z3_small, U_small, V_small,
                                            out_dir, p, frame_duration=500)

print("All composite GIFs for Wind & Z3 analysis have been generated!")